# What is SQL?

SQL (Structured Query Language) is used to:

- Store data
- Retrieve data
- Update data
- Delete data

In simple terms: **SQL helps us manage data in databases**

---

# What is CRUD?

CRUD stands for:

| Operation | Meaning | SQL Command |
| --- | --- | --- |
| C | Create | INSERT |
| R | Read | SELECT |
| U | Update | UPDATE |
| D | Delete | DELETE |

---


## How SQL works

SQL is **declarative**: we describe *what data we want* or *what change we want*, and the database engine decides how to execute it.

For this class, think of the four CRUD operations like this:

| SQL | Purpose | PySpark comparison |
| --- | --- | --- |
| `INSERT` | Add rows | `union()` / `unionByName()` |
| `SELECT` | Read columns/rows | `select()` |
| `UPDATE` | Change column values | `withColumn()` |
| `DELETE` | Remove rows | `filter()` |

The important difference is that SQL works with **tables**, while PySpark commonly works with **DataFrames**.

# 1. CREATE → INSERT Data

### Purpose:

Add new data into a table

### Syntax:

```
INSERT INTO table_name (column1, column2, column3)
VALUES (value1, value2, value3);
```

### Schema and temporary table

A **schema** is a logical container for database objects such as tables and views.

A **temporary table** is session-scoped. It is useful for practice and intermediate work because it is not intended to become a permanent table in the catalog.

For comparison:

- SQL: `CREATE TEMPORARY TABLE customers (...)`
- PySpark: a DataFrame is an in-memory/distributed dataset, and `createOrReplaceTempView()` exposes it to SQL as a temporary view.

In [0]:
create or replace temporary table customers(
    customer_id int,
    name string,
    country string,
    signup_date date
)

In [0]:
select * from customers

In [0]:
insert into customers values(1,'John Smith','UK','2019-01-01')

In [0]:
INSERT INTO customers
VALUES
    (2, 'Alice Brown', 'USA', '2024-02-15'),
    (3, 'Raj Patel', 'India', '2024-03-01'),
    (4, 'Emma Wilson', 'UK', '2024-04-01');

## Key Points:

- Column names must match values
- Data types should be correct
- You can insert multiple rows

## Practice:

- Insert 5 customers from different countries

In [0]:
select * from customers

### Understanding SELECT

`SELECT` is the main SQL command for reading data.

A query normally has two important parts:

```sql
SELECT columns
FROM table;
```

You can think of it as:

- `SELECT` → **which columns do I want?**
- `FROM` → **which table do I want them from?**

This maps closely to PySpark:

```python
df.select("name", "country")
```

# 2. READ → SELECT Data

### Purpose:

Retrieve data from a table

### Syntax:

```
SELECT column1, column2
FROM table_name;
```

In [0]:
-- Filtering Data (WHERE)



### Key Points:

- `*` means all columns
- WHERE filters data
- Conditions matter

### Practice:

- Show all customers
- Show only UK customers
- Show only names

---

### Understanding UPDATE

`UPDATE` changes values in existing rows.

The `SET` clause defines **what changes**, while `WHERE` defines **which rows change**.

This is different from the usual PySpark DataFrame pattern because DataFrames are immutable. In PySpark, we normally create a new column expression and assign the resulting DataFrame back to `df`.

```text
SQL                         PySpark
UPDATE ... SET ... WHERE    withColumn(..., when(...))
```

# 3. UPDATE → Modify Data

### Purpose:

Change existing data

### 🧾 Syntax:


```
UPDATE table_name
SET column=value
WHERE condition;
```


### Important Warning:

```
UPDATE customers SET country='India';
```

👉 This will update **ALL rows**

### Key Points:

- Always use WHERE
- Without WHERE = dangerous

### Practice:

- Update a customer’s country
- Update multiple customers

### Understanding DELETE

`DELETE` removes rows from a table.

The `WHERE` clause is critical because it controls which rows are removed.

In PySpark, the common equivalent is to **keep the rows you want** using `filter()`:

```python
df = df.filter(df.customer_id != 1)
```

So the mental model is:

- SQL: `DELETE ... WHERE condition`
- PySpark: `filter(NOT condition)` / filter the rows you want to keep

# 4. DELETE → Remove Data

### Purpose:

Delete data from table

## 🧾 Syntax:

```
DELETE FROM table_name
WHERE condition;
```

## Example:

## Adding and changing columns

There are two common SQL patterns:

1. **Calculate a column in a query** without changing the table.
2. **Actually add a column to the table** with `ALTER TABLE`, then populate it with `UPDATE`.

For example, `YEAR(signup_date)` calculates the year from a date.

PySpark comparison:

```python
from pyspark.sql.functions import year

df = df.withColumn("signup_year", year("signup_date"))
```

`ALTER TABLE` changes the table schema, whereas `SELECT ... AS` only changes the result of that query.

# Adding columns

# 5. DISTINCT → Remove Duplicate Results

### Purpose

`DISTINCT` returns only unique combinations of the selected columns.

### SQL

```sql
SELECT DISTINCT country
FROM customers;
```

If you select more than one column, uniqueness is checked across the combination of those columns.

### PySpark comparison

```python
df.select("country").distinct()
```

## Practice

- Show the distinct countries.
- Show distinct combinations of `country` and `signup_date`.


# 6. ORDER BY → Sort Data

### Purpose

`ORDER BY` sorts the result.

- `ASC` = ascending (default)
- `DESC` = descending

### SQL

```sql
SELECT *
FROM customers
ORDER BY signup_date DESC;
```

### PySpark comparison

```python
df.orderBy("signup_date")
df.orderBy(df.signup_date.desc())
```

## Practice

- Sort customers by `signup_date` from newest to oldest.
- Sort by `country` ascending and `customer_id` descending.


# 7. WHERE → Multiple Conditions

### Purpose

`WHERE` can combine conditions using `AND`, `OR`, and `NOT`.

### SQL

```sql
SELECT *
FROM customers
WHERE country = 'UK'
   OR country = 'USA';
```

### PySpark comparison

```python
df.filter(
    (df.country == "UK") | (df.country == "USA")
)
```

Remember that PySpark uses `&` and `|` for DataFrame boolean expressions.

## Practice

- Find customers from the UK or USA.
- Find customers whose `customer_id` is greater than 2 and country is India.


# 8. NULL Handling

### Purpose

SQL uses `NULL` to represent a missing/unknown value.

Use `IS NULL` and `IS NOT NULL` rather than `= NULL`.

### SQL

```sql
SELECT *
FROM customers
WHERE country IS NULL;
```

### PySpark comparison

```python
df.filter(df.country.isNull())
df.filter(df.country.isNotNull())
```

For replacing missing values in PySpark, `fillna()` is commonly used.

## Practice

- Find customers where `country` is missing.
- Find customers where `country` is not missing.


# 9. GROUP BY → Aggregate Data

### Purpose

`GROUP BY` groups rows so that aggregate functions can calculate values per group.

Common aggregate functions:

```text
COUNT()
SUM()
AVG()
MIN()
MAX()
```

### SQL

```sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country;
```

### PySpark comparison

```python
from pyspark.sql.functions import count

df.groupBy("country").agg(
    count("*").alias("customer_count")
)
```

## Practice

- Count customers in each country.
- Find the earliest signup date for each country.


# 10. HAVING → Filter Groups

### Purpose

`WHERE` filters **rows before grouping**.

`HAVING` filters **groups after aggregation**.

### SQL

```sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country
HAVING COUNT(*) > 1;
```

### PySpark comparison

PySpark commonly uses `filter()` after the aggregation:

```python
df.groupBy("country")   .count()   .filter("count > 1")
```

## Practice

- Return only countries having more than one customer.


# 11. JOIN → Combine Tables

### Purpose

A `JOIN` combines rows from two tables using a related column.

The most common types are:

| Join | Meaning |
| --- | --- |
| `INNER JOIN` | Matching rows from both tables |
| `LEFT JOIN` | All rows from left + matches from right |
| `RIGHT JOIN` | All rows from right + matches from left |
| `FULL OUTER JOIN` | All rows from both sides |

### SQL

```sql
SELECT c.customer_id, c.name, o.order_id
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id;
```

### PySpark comparison

```python
customers.join(
    orders,
    "customer_id",
    "left"
)
```

## Practice

Assume an `orders` DataFrame/table exists.

- Perform an inner join using `customer_id`.
- Perform a left join and keep all customers.


# 12. UNION → Combine Rows

### Purpose

`UNION` combines the result of two queries **vertically**.

The queries must have compatible column structures.

`UNION` removes duplicate rows.

`UNION ALL` keeps duplicates and is commonly used when you want every row from both inputs.

### PySpark comparison

```python
df1.union(df2)
df1.unionByName(df2)
```

## Practice

- Create two small customer queries with the same columns.
- Combine them using `UNION ALL`.
- Compare the result with `UNION`.


# 13. CAST → Change Data Type

### Purpose

`CAST` converts a value from one data type to another.

### SQL

```sql
SELECT
    CAST(customer_id AS STRING) AS customer_id_string
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import col

df = df.withColumn(
    "customer_id_string",
    col("customer_id").cast("string")
)
```

Casting is especially common when integrating data from different systems.

## Practice

- Cast `customer_id` to `STRING`.
- Cast `signup_date` to `STRING`.


# 14. STRING Functions

### Purpose

SQL provides functions for cleaning and transforming text.

Common functions:

```text
UPPER()
LOWER()
TRIM()
LENGTH()
CONCAT()
SUBSTRING()
REPLACE()
```

### SQL

```sql
SELECT
    UPPER(name) AS name_upper,
    LOWER(country) AS country_lower
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import upper, lower, trim

df = df.withColumn("name_upper", upper("name"))
df = df.withColumn("country_lower", lower("country"))
df = df.withColumn("name_clean", trim("name"))
```

## Practice

- Convert customer names to uppercase.
- Remove leading/trailing spaces from `name`.
- Return the length of each customer name.


# 15. DATE Functions

### Purpose

Date functions are used to extract parts of dates or perform date calculations.

Common functions:

```text
YEAR()
MONTH()
DAY()
DATEDIFF()
DATE_ADD()
DATE_SUB()
CURRENT_DATE()
```

### SQL

```sql
SELECT
    name,
    YEAR(signup_date) AS signup_year,
    MONTH(signup_date) AS signup_month
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import year, month

df = df.withColumn("signup_year", year("signup_date"))
df = df.withColumn("signup_month", month("signup_date"))
```

## Practice

- Show each customer's signup year.
- Show each customer's signup month.
- Calculate the number of days between signup date and today.


# 16. CASE WHEN → Conditional Logic

### Purpose

`CASE WHEN` is SQL's standard way to create conditional values.

### SQL

```sql
SELECT
    name,
    CASE
        WHEN country = 'UK' THEN 'Domestic'
        ELSE 'International'
    END AS customer_type
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import when

df = df.withColumn(
    "customer_type",
    when(df.country == "UK", "Domestic")
    .otherwise("International")
)
```

This is one of the closest SQL ↔ PySpark comparisons.

## Practice

Create a `customer_type` column:

- UK → `Domestic`
- USA/Canada → `North America`
- Everything else → `Other`


# SQL → PySpark Quick Comparison

| SQL | PySpark |
| --- | --- |
| `SELECT` | `select()` |
| `WHERE` | `filter()` / `where()` |
| `DISTINCT` | `distinct()` |
| `ORDER BY` | `orderBy()` |
| `GROUP BY` | `groupBy()` |
| `HAVING` | `filter()` after aggregation |
| `JOIN` | `join()` |
| `UNION ALL` | `union()` |
| `CASE WHEN` | `when().otherwise()` |
| `CAST` | `cast()` |
| `IS NULL` | `isNull()` |
| `COUNT`, `SUM`, `AVG` | `count`, `sum`, `avg` |
| Window functions | `Window` + `.over()` |
| Temporary view | `createOrReplaceTempView()` |
| `LIMIT` | `limit()` |
| `ALTER TABLE` | SQL `ALTER TABLE` / table APIs depending on operation |

### The main idea

If you understand the SQL operation, you can usually learn its PySpark equivalent by asking:

> **What DataFrame transformation produces the same result?**

The syntax changes, but the data-processing concepts remain very similar.
